In [ ]:
# Clona la repository (sostituisci con il tuo username)
!git clone https://github.com/Fabio-Feruglio/Bejing-Air-Quality.git

# Spostati dentro la cartella del progetto
%cd Bejing-Air-Quality

# Installa le dipendenze
!pip install -r requirements.txt

import os, glob, zipfile, urllib.request
import pandas as pd

os.makedirs("data/raw", exist_ok=True)

url = "https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip"
urllib.request.urlretrieve(url, "beijing.zip")

with zipfile.ZipFile("beijing.zip") as z:
    z.extractall("beijing_raw")

# lo zip esterno contiene un secondo zip con i 12 CSV per stazione
inner_zip = glob.glob("beijing_raw/**/*.zip", recursive=True)
if inner_zip:
    with zipfile.ZipFile(inner_zip[0]) as z:
        z.extractall("beijing_raw")

station_files = glob.glob("beijing_raw/**/PRSA_Data_*.csv", recursive=True)
print(f"Trovati {len(station_files)} file per stazione")

df_all = pd.concat([pd.read_csv(f) for f in station_files], ignore_index=True)
df_all.to_csv("data/raw/Beijing_Multisite_air_Quality_data.csv", index=False)
print(df_all.shape, df_all['station'].unique())

In [ ]:
# Questo aprirà un prompt interattivo per inserire la tua chiave API di wandb
!wandb login

from google.colab import userdata
import wandb

wandb.login(key=userdata.get('WANDB_API_KEY'))

In [ ]:
!python run_experiment.py \
    --data_path "data/raw/Beijing_Multisite_air_Quality_data.csv" \
    --epochs 2 \
    --batch_size 128 \
    --val_size 0.15 \
    --test_size 0.15 \
    --run_name "test_veloce_colab"